# Early Stopping

## Learning Objectives
1. Understand why early stopping prevents overfitting by halting training when validation loss stops improving.
2. Implement a patience-based early stopping mechanism from scratch with numpy.
3. Build a PyTorch EarlyStopping class that restores the best model weights.
4. Apply early stopping with learning-rate scheduling and model checkpointing in production training loops.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Basic Early Stopping (NumPy Simulation)

In [ ]:
# Simulate a training/validation loss curve and apply patience-based early stopping
def simulate_losses(n_epochs=100, noise_std=0.05):
    """Generate synthetic train/val losses where val loss starts rising after epoch ~40."""
    epochs = np.arange(n_epochs)
    train_loss = 1.0 / (1 + 0.1 * epochs) + np.random.normal(0, noise_std, n_epochs)
    # Validation loss has a U-shape: improves then degrades (overfitting)
    val_loss = (1.0 / (1 + 0.1 * epochs) + 0.003 * (epochs - 40) ** 2 / 40
                + np.random.normal(0, noise_std, n_epochs))
    return train_loss, val_loss


def early_stop(val_losses, patience=5, min_delta=1e-4):
    """Return the epoch index at which training should stop."""
    best_loss = float("inf")
    wait = 0
    for epoch, loss in enumerate(val_losses):
        if loss < best_loss - min_delta:
            best_loss = loss
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            return epoch
    return len(val_losses) - 1


train_loss, val_loss = simulate_losses(100)
stop_epoch = early_stop(val_loss, patience=5)
print(f"Training stopped at epoch {stop_epoch} (out of 100)")
print(f"Best val loss: {val_loss[:stop_epoch+1].min():.4f}")
print(f"Val loss at epoch 100 would be: {val_loss[-1]:.4f}")


## Level 2: PyTorch EarlyStopping with Best-Weight Restoration

In [ ]:
class EarlyStopping:
    """Patience-based early stopper that restores the best model weights."""

    def __init__(self, patience: int = 5, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss: float = float("inf")
        self.wait: int = 0
        self.best_weights = None
        self.stopped_epoch: int = 0

    def step(self, val_loss: float, model: nn.Module) -> bool:
        """Return True when training should stop."""
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.wait = 0
            # Deep copy state_dict so later updates don't overwrite it
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.wait += 1
        if self.wait >= self.patience:
            return True
        return False

    def restore(self, model: nn.Module) -> None:
        """Load the best-checkpoint weights back into the model."""
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


# --- Build a tiny MLP on synthetic regression data ---
X = torch.randn(500, 10, device=device)
true_w = torch.randn(10, 1, device=device)
y = X @ true_w + 0.1 * torch.randn(500, 1, device=device)

split = 400
train_ds = TensorDataset(X[:split], y[:split])
val_ds = TensorDataset(X[split:], y[split:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)


def build_model():
    return nn.Sequential(
        nn.Linear(10, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 1),
    ).to(device)


model = build_model()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
stopper = EarlyStopping(patience=8, min_delta=1e-5)

history = {"train": [], "val": []}
for epoch in range(200):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimiser.zero_grad()
        try:
            loss = criterion(model(xb), yb)
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM — reduce batch size")
                torch.cuda.empty_cache()
                continue
            raise
        loss.backward()
        optimiser.step()
        running_loss += loss.item() * len(xb)
    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        val_loss = sum(
            criterion(model(xb), yb).item() * len(xb)
            for xb, yb in val_loader
        ) / len(val_loader.dataset)

    history["train"].append(train_loss)
    history["val"].append(val_loss)

    if stopper.step(val_loss, model):
        print(f"Early stop at epoch {epoch+1} — restoring best weights")
        stopper.restore(model)
        break

print(f"Final val MSE (best checkpoint): {stopper.best_loss:.6f}")


## Real-World Example 1: Early Stopping + ReduceLROnPlateau

In [ ]:
# Combine early stopping with learning-rate reduction on plateau.
# Strategy: reduce LR when val_loss stalls for `factor_patience` steps;
#           stop entirely when LR drops below min_lr.

model_rw1 = build_model()
opt_rw1 = torch.optim.Adam(model_rw1.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_rw1, mode="min", factor=0.5, patience=3, min_lr=1e-6, verbose=False
)
stopper_rw1 = EarlyStopping(patience=10, min_delta=1e-6)
MIN_LR = 1e-6
criterion_rw1 = nn.MSELoss()

lr_history = []
val_history = []
for epoch in range(300):
    model_rw1.train()
    for xb, yb in train_loader:
        opt_rw1.zero_grad()
        criterion_rw1(model_rw1(xb), yb).backward()
        opt_rw1.step()

    model_rw1.eval()
    with torch.no_grad():
        vl = sum(
            criterion_rw1(model_rw1(xb), yb).item() * len(xb)
            for xb, yb in val_loader
        ) / len(val_loader.dataset)

    scheduler.step(vl)
    current_lr = opt_rw1.param_groups[0]["lr"]
    lr_history.append(current_lr)
    val_history.append(vl)

    # Stop when LR can't decrease further — no more learning to be done
    if current_lr <= MIN_LR:
        print(f"LR reached minimum at epoch {epoch+1}. Stopping.")
        stopper_rw1.restore(model_rw1)
        break

    if stopper_rw1.step(vl, model_rw1):
        print(f"Early stop at epoch {epoch+1}")
        stopper_rw1.restore(model_rw1)
        break

print(f"Final LR: {lr_history[-1]:.2e}  |  Best val MSE: {min(val_history):.6f}")


## Real-World Example 2: Model Checkpointing on Improvement

In [ ]:
import os
import tempfile

# Save a checkpoint every time val_loss improves — production pattern
# for long training runs where you may need to resume or roll back.

CKPT_DIR = tempfile.mkdtemp()
model_rw2 = build_model()
opt_rw2 = torch.optim.Adam(model_rw2.parameters(), lr=1e-3)
criterion_rw2 = nn.MSELoss()
best_val_loss = float("inf")
checkpoint_epochs = []

for epoch in range(60):
    model_rw2.train()
    for xb, yb in train_loader:
        opt_rw2.zero_grad()
        criterion_rw2(model_rw2(xb), yb).backward()
        opt_rw2.step()

    model_rw2.eval()
    with torch.no_grad():
        vl = sum(
            criterion_rw2(model_rw2(xb), yb).item() * len(xb)
            for xb, yb in val_loader
        ) / len(val_loader.dataset)

    if vl < best_val_loss:
        best_val_loss = vl
        ckpt_path = os.path.join(CKPT_DIR, f"best_epoch_{epoch+1:03d}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model_rw2.state_dict(),
            "optimizer_state_dict": opt_rw2.state_dict(),
            "val_loss": vl,
        }, ckpt_path)
        checkpoint_epochs.append(epoch + 1)

print(f"Saved {len(checkpoint_epochs)} checkpoints at epochs: {checkpoint_epochs}")
print(f"Best val MSE: {best_val_loss:.6f}")

# Restore from best checkpoint
ckpt = torch.load(ckpt_path, map_location=device)
model_rw2.load_state_dict(ckpt["model_state_dict"])
print(f"Restored checkpoint from epoch {ckpt['epoch']} with val_loss={ckpt['val_loss']:.6f}")


## Real-World Example 3: Learning Curve Comparison — Early Stop vs Full Training

In [ ]:
# Demonstrate that early stopping prevents overfitting and generalises better
# than running for the full epoch budget.

def train_model(n_epochs, use_early_stop, patience=8):
    """Train model and return (train_hist, val_hist, stopped_at)."""
    m = build_model()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    stopper = EarlyStopping(patience=patience) if use_early_stop else None
    train_hist, val_hist = [], []
    stopped_at = n_epochs

    for epoch in range(n_epochs):
        m.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            crit(m(xb), yb).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            tl = sum(crit(m(xb), yb).item() * len(xb) for xb, yb in train_loader) / len(train_loader.dataset)
            vl = sum(crit(m(xb), yb).item() * len(xb) for xb, yb in val_loader) / len(val_loader.dataset)
        train_hist.append(tl)
        val_hist.append(vl)
        if stopper and stopper.step(vl, m):
            stopped_at = epoch + 1
            stopper.restore(m)
            break

    return train_hist, val_hist, stopped_at


hist_es_train, hist_es_val, stop_at = train_model(200, use_early_stop=True)
hist_full_train, hist_full_val, _ = train_model(200, use_early_stop=False)

print(f"Early stop at epoch {stop_at}  |  best val MSE: {min(hist_es_val):.6f}")
print(f"Full training (200 ep)        |  final val MSE: {hist_full_val[-1]:.6f}")
print(f"Val MSE improvement: {hist_full_val[-1] - min(hist_es_val):.6f}")

# Early stopping: fewer epochs, less overfitting, often better or equal generalisation
